In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
import os

In [10]:
from dotenv import load_dotenv, set_key
import os

dotenv_path = ".env"

def save_api_keys(groq_api_key: str, pinecone_api_key: str):
    set_key(dotenv_path, "GROQ_API_KEY", groq_api_key)
    set_key(dotenv_path, "PINECONE_API_KEY", pinecone_api_key)
    print("API keys saved successfully to .env")


In [11]:
load_dotenv(".env", override=True)

GROQ_API_KEY     = os.getenv("GROQ_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX   = "medical-chatbot"
PINECONE_ENV     = "us-east-1"

print("All variables loaded")

All variables loaded


In [12]:

# BUILT-IN MEDICAL TEXTS
MEDICAL_TEXTS = [
    """Diabetes Mellitus: Type 1 (autoimmune, insulin deficiency) and Type 2 (insulin resistance).
    Symptoms: polyuria, polydipsia, polyphagia, weight loss, fatigue, blurred vision, slow healing.
    Treatment T1: insulin therapy. T2: metformin, GLP-1 agonists, SGLT-2 inhibitors.
    Complications: nephropathy, neuropathy, retinopathy, cardiovascular disease. HbA1c target <7%.""",

    """Hypertension: BP more than 130/80 mmHg. Often asymptomatic (silent killer).
    Lifestyle: DASH diet, exercise, reduce sodium less than 2.3g/day, quit smoking, limit alcohol.
    Medications: ACE inhibitors, ARBs, calcium channel blockers, thiazide diuretics, beta-blockers.
    Complications: stroke, heart attack, heart failure, kidney damage.""",

    """Fever: Temperature more than 38C. Causes: bacterial/viral infections, inflammation.
    Treatment: paracetamol 500-1000mg every 4-6h, ibuprofen 200-400mg every 6-8h, rest, fluids.
    Seek emergency: fever more than 39.4C, stiff neck, rash, breathing difficulty, confusion.""",

    """Common Cold: Viral infection (rhinovirus). Symptoms: runny nose, sneezing, sore throat, cough.
    Duration 7-10 days. Treatment: symptomatic only - rest, fluids, decongestants, saline spray.
    Antibiotics NOT effective. Prevention: handwashing, avoid face touching.""",

    """Asthma: Chronic airway inflammation. Symptoms: wheezing, breathlessness, chest tightness, cough.
    Triggers: allergens, exercise, cold air, smoke, infections, stress.
    Acute relief: salbutamol inhaler 2-4 puffs. Long-term: ICS (beclomethasone).
    Every patient needs a written Asthma Action Plan.""",

    """Migraine: Unilateral throbbing headache, nausea, photophobia, phonophobia. Aura in 25%.
    Triggers: stress, hormones, certain foods, sleep changes, bright lights.
    Acute: NSAIDs (ibuprofen 400-600mg), triptans (sumatriptan). Prevention: propranolol, topiramate.
    Red flags: thunderclap headache, fever and stiff neck, neurological deficits - emergency.""",

    """COVID-19: Respiratory illness, transmission via droplets/aerosols.
    Symptoms: fever, dry cough, fatigue, loss of taste/smell, shortness of breath.
    High risk: elderly, diabetes, hypertension, immunocompromised.
    Prevention: vaccination, masking, ventilation, handwashing. Long COVID: more than 12 weeks symptoms.""",

    """CPR: Call 108/112. 30 chest compressions (5-6cm deep, 100-120/min) then 2 rescue breaths.
    AED: turn on, attach pads, analyse, shock if advised, resume CPR immediately.
    Choking: 5 back blows then 5 abdominal thrusts (Heimlich). If unconscious - CPR.
    Burns: cool with running water 20 minutes. NOT ice, butter, or toothpaste.""",

    """Depression: Low mood, loss of interest, fatigue, sleep/appetite changes for 2+ weeks.
    Treatment: CBT, SSRIs (fluoxetine, sertraline), lifestyle (exercise, sleep, social support).
    Anxiety: excessive worry, restlessness, irritability. Treatment: CBT, SSRIs/SNRIs, buspirone.
    Crisis India: iCall 9152987821 | Vandrevala 1860-2662-345 | AASRA 9820466627""",

    """Nutrition: Fruits and vegetables 5+ portions/day, whole grains, lean protein, healthy fats.
    Limit: saturated fat, trans fat, sugar, sodium less than 5g/day, processed foods.
    Exercise: 150+ min moderate aerobic/week + 2 strength sessions. Sleep 7-9 hours.
    BMI: less than 18.5 underweight | 18.5-24.9 normal | 25-29.9 overweight | 30+ obese.""",
]

# PDF LOADER
def load_pdfs(folder="./"):
    docs = []
    if os.path.exists(folder):
        for filename in os.listdir(folder):
            if filename.endswith(".pdf"):
                full_path = os.path.join(folder, filename)
                try:
                    print(f"   Loading: {filename}...")
                    loader = PyPDFLoader(full_path)
                    pages  = loader.load()
                    docs.extend(pages)
                    print(f"   {filename} loaded - {len(pages)} pages")
                except Exception as e:
                    print(f"   Error loading {filename}: {e}")
    return docs

# LOAD DOCUMENTS
print("Loading Medical Knowledge Base...")

pdf_docs = load_pdfs(folder="./")

if pdf_docs:
    documents = pdf_docs
    print(f"PDF Mode - Using Medical_book.pdf!")
    print(f"Total pages loaded: {len(pdf_docs)}")
else:
    print("No PDF found - Using built-in medical knowledge")
    documents = [
        Document(
            page_content=text,
            metadata={"source": "builtin_knowledge", "topic": i}
        )
        for i, text in enumerate(MEDICAL_TEXTS)
    ]
    print(f"   Total built-in topics: {len(documents)}")

# SPLIT INTO CHUNKS
print("Splitting into chunks...")

splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 500,
    chunk_overlap = 50,
    separators    = ["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")
print("Knowledge Base Ready!")

Loading Medical Knowledge Base...
   Loading: Medical_book.pdf...
   Medical_book.pdf loaded - 637 pages
PDF Mode - Using Medical_book.pdf!
Total pages loaded: 637
Splitting into chunks...
Total chunks created: 5961
Knowledge Base Ready!


In [13]:
import warnings
warnings.filterwarnings("ignore")
import os
from dotenv import load_dotenv

# LOAD ENV VARIABLES
load_dotenv(".env", override=True)
GROQ_API_KEY     = os.getenv("GROQ_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX   = "medical-chatbot"

print(f"GROQ Key loaded: {'' if GROQ_API_KEY else ' MISSING'}")
print(f"Pinecone Key loaded: {'' if PINECONE_API_KEY else ' MISSING'}")

# RECONNECT TO PINECONE (no re-uploading) 
from pinecone import Pinecone
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore

print("\nLoading embeddings...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)
print("Embeddings Ready ")

print("Connecting to Pinecone...")
pc    = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX)
print(f"Connected to index: '{PINECONE_INDEX}' ")

vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    text_key="text"
)
print("Vectorstore Ready ")

# BUILD LLM + RAG CHAIN 
from langchain_groq import ChatGroq
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferWindowMemory
from langchain.prompts import PromptTemplate

SYSTEM_PROMPT = """
You are MedBot, an expert AI medical assistant.
Use the provided context to give accurate, empathetic medical information.
Always recommend consulting a qualified healthcare professional for diagnosis.
Never provide dosage recommendations without mentioning 'consult your doctor'.
Medical Context: {context}
Conversation History: {chat_history}
"""

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3,
    max_tokens=1024,
    groq_api_key=GROQ_API_KEY
)
print("LLM: Groq LLaMA3 Ready ")

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)
print("Retriever Ready")

memory = ConversationBufferWindowMemory(
    k=5,
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)
print("Memory Ready")

prompt = PromptTemplate(
    input_variables=["context", "chat_history", "question"],
    template=SYSTEM_PROMPT + "\nPatient Question: {question}\n\nMedBot:"
)

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
    verbose=False
)
print("RAG Chain Ready")

# QUICK TEST
print("\nTesting...")
test = qa_chain.invoke({"question": "What are symptoms of diabetes?"})
print(f"Test Answer: {test['answer'][:150]}...")
print("\nEverything is working! Run the Gradio cell now.")

GROQ Key loaded: 
Pinecone Key loaded: 

Loading embeddings...
Embeddings Ready 
Connecting to Pinecone...
Connected to index: 'medical-chatbot' 
Vectorstore Ready 
LLM: Groq LLaMA3 Ready 
Retriever Ready
Memory Ready
RAG Chain Ready

Testing...
Test Answer: According to the provided context, symptoms of diabetes mellitus (Type I) include:

1. Fatigue
2. Hyperglycemia (abnormally high level of glucose in t...

Everything is working! Run the Gradio cell now.


In [14]:
import gradio as gr

def chat(user_message, history):
    if not user_message.strip():
        return "", history
    try:
        result     = qa_chain.invoke({"question": user_message})
        bot_answer = result["answer"]
    except Exception as e:
        bot_answer = f"Error: {str(e)}"
    history = history + [[user_message, bot_answer]]
    return "", history

def clear_chat():
    try:
        memory.clear()
    except:
        pass
    return "", []

def make_topic_fn(q):
    def topic_fn(history):
        return chat(q, history or [])
    return topic_fn

QUICK_TOPICS = [
    ("Diabetes",       "What are diabetes symptoms and treatment?"),
    ("Hypertension",   "How to manage high blood pressure?"),
    ("Common Cold",    "Remedies for common cold?"),
    ("Asthma",         "Asthma symptoms, triggers, treatment?"),
    ("Headache",       "Migraine causes and treatment?"),
    ("COVID-19",       "COVID-19 symptoms and precautions?"),
    ("Mental Health",  "Tell me about depression and anxiety."),
    ("Nutrition",      "Healthy diet and nutrition tips."),
    ("First Aid CPR",  "How to perform CPR?"),
    ("Fever",          "When to worry about fever?"),
    ("Medications",    "Common medications and uses."),
    ("Sleep",          "How much sleep do adults need?"),
]

with gr.Blocks(title="MedBot") as demo:

    gr.Markdown(
        "# MedBot - AI Medical Assistant\n"
        "### Powered by LangChain - Pinecone - Groq LLaMA3 - RAG"
    )

    gr.HTML(
        '<div style="background:#fdecea; border:1px solid #f5c6cb;'
        'border-radius:8px; padding:10px 15px; color:#c0392b;'
        'font-size:13px; margin-bottom:10px;">'
        "DISCLAIMER: General health information only. "
        "NOT a substitute for professional medical diagnosis. "
        "Always consult a qualified doctor."
        "</div>"
    )

    with gr.Row():
        with gr.Column(scale=1, min_width=180):
            gr.Markdown("### Quick Topics")
            topic_btns = []
            for label, _ in QUICK_TOPICS:
                btn = gr.Button(label, size="sm")
                topic_btns.append(btn)
            gr.Markdown("---")
            gr.Markdown(
                "Powered By\n"
                "- LangChain\n"
                "- Pinecone\n"
                "- Groq LLaMA3\n"
                "- RAG Pipeline"
            )

        with gr.Column(scale=4):
            chatbot = gr.Chatbot(
                label="Medical Chat",
                height=480
            )
            with gr.Row():
                msg_input = gr.Textbox(
                    placeholder="Ask a medical question here...",
                    label="Your Question",
                    scale=5,
                    lines=2
                )
                with gr.Column(scale=1, min_width=120):
                    send_btn = gr.Button(
                        "Send",
                        variant="primary",
                        size="lg"
                    )
                    clear_btn = gr.Button(
                        "Clear",
                        variant="secondary",
                        size="sm"
                    )

    send_btn.click(
        fn=chat,
        inputs=[msg_input, chatbot],
        outputs=[msg_input, chatbot]
    )
    msg_input.submit(
        fn=chat,
        inputs=[msg_input, chatbot],
        outputs=[msg_input, chatbot]
    )
    clear_btn.click(
        fn=clear_chat,
        outputs=[msg_input, chatbot]
    )
    for btn, (_, query) in zip(topic_btns, QUICK_TOPICS):
        btn.click(
            fn=make_topic_fn(query),
            inputs=[chatbot],
            outputs=[msg_input, chatbot]
        )

print("Launching MedBot...")
demo.launch(share=True)
print("MedBot is running!")


Launching MedBot...
Running on local URL:  http://127.0.0.1:7861
IMPORTANT: You are using gradio version 3.50.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://506c292087b54ac581.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


MedBot is running!
